In [ ]:
!pip install python-telegram-bot

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from google.colab.patches import cv2_imshow

In [ ]:
!pip install ultralytics supervision roboflow
import ultralytics
ultralytics.checks()

In [ ]:
from IPython.display import Image as IPyImage

In [ ]:
import glob
import os
from IPython.display import Image as IPyImage, display

#latest_folder = max(glob.glob(f'runs/detect/predict*/'), key=os.path.getmtime)
#for img in glob.glob(f'{latest_folder}/*.jpg')[:3]:
#    display(IPyImage(filename=img, width=600))
 #   print("\n")

# Extracting ROI and Getting Color

# Multi ROIS

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt


def person_detect(image, model_path, dataset="None"):
    """Draw bounding boxes on detected objects with color labels."""
    person_count = 0
    person_roi_dict = {}
    
    try:
        model = YOLO(model_path)
        
        results = model.predict(image)[0]

        if not results.boxes:
            print("No objects detected.")
            return image
        
        

        for box in results.boxes:
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            if x1 >= 0 and y1 >= 0 and x2 <= image.shape[1] and y2 <= image.shape[0] and x2 > x1 and y2 > y1:
                person_boxes = [(x1, y1, x2-x1, y2-y1)]
                person_roi_dict[person_count] = person_boxes
                person_count += 1
                
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)
                label = f"({conf:.2f})"
                label_position = (x1, y1 - 10) if y1 > 20 else (x1, y1 + 20)
                cv2.putText(image, label, label_position, cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 2)
                    
            else:
                print(f"Invalid bounding box coordinates: {x1}, {y1}, {x2}, {y2} for image shape {image.shape}")
                continue
        return person_roi_dict, image

    except Exception as e:
        print(f"An error occurred: {e}")
        return None, None

In [ ]:
import cv2
import numpy as np

def get_hardhat_color(helmet_roi):
    """
    Optimized to detect ONLY yellow or white hardhats in real-world conditions.
    Returns: "yellow", "white", or "unknown" (if not a valid hardhat color).
    
    Features:
    - Uses saturation-weighted hue for accurate color dominance
    - Masks out shadows and glare
    - Strict safety-color ranges (rejects orange, green, etc.)
    - Handles lighting variations (sun, shade, indoor)
    """
    hsv = cv2.cvtColor(helmet_roi, cv2.COLOR_BGR2HSV)
    
    # Mask out shadows (V < 50) and extreme glare (S < 10, V > 240)
    mask = cv2.inRange(hsv, (0, 10, 50), (255, 255, 240))
    masked_hsv = cv2.bitwise_and(hsv, hsv, mask=mask)
    
    # Extract non-masked pixels
    pixels = masked_hsv.reshape(-1, 3)
    pixels = pixels[np.all(pixels != [0, 0, 0], axis=1)]
    
    if len(pixels) == 0:
        return "unknown"
    
    # Weighted average (saturation = importance)
    h, s, v = pixels[:, 0], pixels[:, 1], pixels[:, 2]
    dominant_hue = np.average(h, weights=s)
    mean_sat = np.mean(s)
    mean_val = np.mean(v)
    
    # Classify
    return classify_yellow_or_white(dominant_hue, mean_sat, mean_val)

def classify_yellow_or_white(h, s, v):
    """
    Strict classification for safety hardhats (yellow/white only).
    Rejects all other colors including orange, green, blue, etc.
    """
    # OpenCV HSV ranges: H(0-180), S(0-255), V(0-255)
    
    # 🔹 WHITE: Low saturation + high brightness
    if v > 200 and s < 50:
        return "white"
    
    # 🔹 YELLOW: Hardhat-specific range (rejects orange)
    if 18 <= h <= 42:  # Tightened hue range for true yellows
        if s >= 40 and v >= 100:  # Minimum saturation/value thresholds
            # Reject orange (high saturation + low value)
            if not (h <= 25 and s > 160 and v < 150):
                return "yellow"
    
    return "unknown"

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt

def get_color_from_hsv(helmet_roi):
    """Use histogram-based detection for the dominant color in HSV."""
    hsv = cv2.cvtColor(helmet_roi, cv2.COLOR_BGR2HSV)
    
    # Calculate histogram in HSV space
    hist_hue = cv2.calcHist([hsv], [0], None, [256], [0, 256])
    hist_saturation = cv2.calcHist([hsv], [1], None, [256], [0, 256])
    hist_value = cv2.calcHist([hsv], [2], None, [256], [0, 256])
    
    dominant_hue = np.argmax(hist_hue)
    dominant_saturation = np.argmax(hist_saturation)
    dominant_value = np.argmax(hist_value)

    return dominant_hue, dominant_saturation, dominant_value


def get_color_name(h, s, v):
    """
    Optimized for yellow hardhats in real conditions.
    Returns only: 'yellow', 'white', 'unknown'
    - Handles sunlight, shadows, white-yellowish surfaces
    - Rejects orange, green, and false yellows
    - Based on real hardhat HSV data
    """
    # 🔹 WHITE: Bright and low-to-moderate saturation (handles yellowish-white under warm light)
    if v > 200 and s < 60:
        return "white"

    # 🔹 YELLOW: Hardhat yellow range (adjusted for real data)
    # Typical hardhat: H=20–40 (OpenCV scale), but can shift in sun/shade
    if (18 <= h <= 48) and (s >= 50) and (v >= 100):
        # Allow lower saturation because worn hats fade
        # Allow lower V because of shadows
        return "yellow"

    return "unknown"

def helmet_crop(image, helmet_boxes):
    """Crop the image focusing on the middle part of the upper half of the bounding box."""
    helmet_images = []
    for (x, y, w, h) in helmet_boxes:
        # Define the middle part of the upper half
        mid_x1 = x + w // 4  # Start from 1/4th width
        mid_x2 = x + (3 * w) // 4  # End at 3/4th width
        mid_y1 = y  # Start from the top
        mid_y2 = y + h // 2  # End at half the height

        middle_upper_part = image[mid_y1:mid_y2, mid_x1:mid_x2]
        middle_upper_part = reduce_glare(middle_upper_part)
        helmet_images.append(middle_upper_part)

    return helmet_images


def draw_bounding_boxes_with_color_better(image, model_path, dataset="None"):
    """Draw bounding boxes on detected objects with color labels."""
    try:
        model = YOLO(model_path)
        
        results = model.predict(image,max_det=1)[0]

        #image = cv2.imread(image_path)
        
        #if image is None:
        #    raise FileNotFoundError(f"Image at path {image_path} could not be loaded.")

        if not results.boxes:
            print("No objects detected.")
            return "none", image

        color_counts = defaultdict(int)

        for box in results.boxes:
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            if x1 >= 0 and y1 >= 0 and x2 <= image.shape[1] and y2 <= image.shape[0] and x2 > x1 and y2 > y1:
                helmet_boxes = [(x1, y1, x2-x1, y2-y1)]
                helmet_images = helmet_crop(image, helmet_boxes)

                for helmet_roi in helmet_images:
                    if helmet_roi.size == 0:
                        print("Empty helmet ROI. Skipping.")
                        continue

                    # Get the dominant color using histogram-based method
                    #dominant_hue, dominant_saturation, dominant_value = get_color_from_hsv(helmet_roi)
                    #color_name = get_color_name(dominant_hue, dominant_saturation, dominant_value)
                    color_name = get_hardhat_color(helmet_roi)
                    color_counts[color_name] += 1
                    

                    if color_name != "unknown":
                        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        label = f"{color_name} ({conf:.2f})"
                        label_position = (x1, y1 - 10) if y1 > 20 else (x1, y1 + 20)
                        cv2.putText(image, label, label_position, cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 2)
    
                        print(f"Detected helmet color: {color_name} | Confidence: {conf:.2f}")
            else:
                print(f"Invalid bounding box coordinates: {x1}, {y1}, {x2}, {y2} for image shape {image.shape}")
                continue

        #cv2_imshow(image)
        #cv2.imwrite(dataset,image)
        
            
        return color_name, image

    except Exception as e:
        print(f"An error occurred: {e}")
        return "none", image

# Mobile Detect

In [ ]:
import cv2
import numpy as np

def reduce_glare(image, threshold=220, blur_ksize=15):
    """
    Reduces glare from shiny surfaces (like helmets) in sunny conditions.
    
    Parameters:
    - image: input BGR image (np.ndarray)
    - threshold: brightness threshold to detect glare (0–255)
    - blur_ksize: blur kernel size to blend glare region
    
    Returns:
    - glare-reduced image
    """
    # Convert to grayscale to detect bright regions
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Create glare mask (bright regions)
    _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

    # Optional: dilate mask to cover edges
    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)

    # Inpaint glare region
    result = cv2.inpaint(image, mask, inpaintRadius=5, flags=cv2.INPAINT_TELEA)

    return result

In [ ]:
import cv2
import numpy as np

def helmet_crop(image, helmet_boxes):
    """Crop the image focusing on the middle part of the upper half of the bounding box."""
    helmet_images = []
    for (x, y, w, h) in helmet_boxes:
        # Define the middle part of the upper half
        mid_x1 = x + w // 4  # Start from 1/4th width
        mid_x2 = x + (3 * w) // 4  # End at 3/4th width
        mid_y1 = y  # Start from the top
        mid_y2 = y + h // 2  # End at half the height

        middle_upper_part = image[mid_y1:mid_y2, mid_x1:mid_x2]
        middle_upper_part = reduce_glare(middle_upper_part)
        helmet_images.append(middle_upper_part)

    return helmet_images




def draw_bounding_boxes_mobile(image_path, detect_hat_image, model_path, dataset="None"):
    """Draw bounding boxes on detected objects with color labels."""
    try:
        model = YOLO(model_path)
        results = model.predict(image_path, max_det=1)[0]

        count = 0
        
        image = detect_hat_image
        if image is None:
            raise FileNotFoundError(f"Image at path {image_path} could not be loaded.")

        if not results.boxes:
            print("No objects detected.")
            return count, image

        

        for box in results.boxes:
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            if x1 >= 0 and y1 >= 0 and x2 <= image.shape[1] and y2 <= image.shape[0] and x2 > x1 and y2 > y1:
                count += 1
                helmet_boxes = [(x1, y1, x2-x1, y2-y1)]
                helmet_images = helmet_crop(image, helmet_boxes)

                for helmet_roi in helmet_images:
                    if helmet_roi.size == 0:
                        print("Empty Mobile ROI. Skipping.")
                        continue

                    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    #label = "Detected"
                    label = f"Detected ({conf:.2f})"
                    label_position = (x1, y1 - 10) if y1 > 20 else (x1, y1 + 20)
                    cv2.putText(image, label, label_position, cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 2)
            else:
                label = "Not Detected"
                label_position = (x1, y1 - 10) if y1 > 20 else (x1, y1 + 20)
                cv2.putText(image, label, label_position, cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 2)
                print(f"Invalid bounding box coordinates: {x1}, {y1}, {x2}, {y2} for image shape {image.shape}")
                return count , image
                

        #cv2_imshow(image)
        #cv2.imwrite(dataset,image)
        
            
        return count , image

    except Exception as e:
        print(f"An error occurred: {e}")
        return 0, None

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

def is_mobile_being_used(image_path, angle_threshold=118):
    """
    Detects if a person is using a mobile phone based on YOLO pose & phone detection.
    Scale-invariant version (no fixed pixel thresholds).
    """
    # --- Load image ---
    img = cv2.imread(image_path) if isinstance(image_path, str) else image_path
    if img is None:
        raise ValueError("Image not found.")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- Load models ---
    pose_model = YOLO("/kaggle/input/yolo11spose/yolo11s-pose.pt")
    phone_model = YOLO("/kaggle/input/mobiledetect/mobile detect/best.pt")

    # --- Pose detection ---
    pose_results = pose_model(img_rgb)
    keypoints = pose_results[0].keypoints.data.cpu().numpy()
    if len(keypoints) == 0:
        return "No Person Detected", img
    kps = keypoints[0]

    # --- Extract relevant joints ---
    left_wrist, right_wrist = kps[9][:2], kps[10][:2]
    left_elbow, right_elbow = kps[7][:2], kps[8][:2]
    left_shoulder, right_shoulder = kps[5][:2], kps[6][:2]

    # --- Phone detection ---
    phone_results = phone_model(img_rgb)
    boxes = phone_results[0].boxes.xyxy.cpu().numpy()
    if len(boxes) == 0:
        return "Mobile Not Detected", img

    phone_box = boxes[0]
    phone_center = np.array([
        (phone_box[0] + phone_box[2]) / 2,
        (phone_box[1] + phone_box[3]) / 2
    ])
    conf = phone_results[0].boxes.conf[0].item()

    # --- Draw phone box ---
    cv2.rectangle(img, (int(phone_box[0]), int(phone_box[1])), (int(phone_box[2]), int(phone_box[3])), (0, 0, 255), 2)
    cv2.putText(img, f"Phone ({conf:.2f})", 
                (int(phone_box[0]), int(phone_box[1]) - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # --- Helper: angle between joints ---
    def angle(a, b, c):
        a, b, c = np.array(a), np.array(b), np.array(c)
        ba, bc = a - b, c - b
        cosang = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
        return np.degrees(np.arccos(np.clip(cosang, -1.0, 1.0)))

    # --- Compute arm angles ---
    left_angle = angle(left_wrist, left_elbow, left_shoulder)
    right_angle = angle(right_wrist, right_elbow, right_shoulder)
    left_possible = left_angle < angle_threshold
    right_possible = right_angle < angle_threshold

    # --- Determine body scale (shoulder width) ---
    shoulder_width = np.linalg.norm(right_shoulder - left_shoulder)
    if shoulder_width < 1e-3:
        shoulder_width = 1.0

    # --- Compute normalized wrist-to-phone distances ---
    dist_left = np.linalg.norm(phone_center - left_wrist) / shoulder_width
    dist_right = np.linalg.norm(phone_center - right_wrist) / shoulder_width

    # --- Decision (scale-invariant) ---
    # Empirically, if wrist-phone distance < 0.8×shoulder width → likely using phone
    threshold = 0.8
    left_close = left_possible and dist_left < threshold
    right_close = right_possible and dist_right < threshold

    using_phone = "Mobile Used" if (left_close or right_close) else "Mobile Un-Used"

    # --- Visualization ---
    def draw_line(p1, p2, color):
        cv2.line(img, tuple(map(int, p1)), tuple(map(int, p2)), color, 3)

    if left_close:
        draw_line(left_wrist, left_elbow, (255, 255, 0))
        draw_line(left_elbow, left_shoulder, (255, 255, 0))
        draw_line(left_wrist, phone_center, (255, 255, 0))
    if right_close:
        draw_line(right_wrist, right_elbow, (0, 255, 255))
        draw_line(right_elbow, right_shoulder, (0, 255, 255))
        draw_line(right_wrist, phone_center, (0, 255, 255))

    return using_phone, img

In [ ]:
import cv2
import gc

def hardhat_mobile_detect(image_path, person_model_path, helmet_model_path, mobile_model_path):
    # Load the image
    count = 0
    
    if type(image_path) == str:
        image = cv2.imread(image_path)
    else:
        image = image_path
        
    if image is None:
        raise FileNotFoundError(f"Image not found at {image_path}")

    # Detect persons
    person_roi_dict, full_image = person_detect(image=image, model_path=person_model_path, dataset="None")

    # Process each detected person
    for rois in person_roi_dict.values():
        if not rois:
            continue
        x1, y1, width, height = rois[0]
        
        # Calculate bottom-right coordinates
        x2, y2 = x1 + width, y1 + height

        # Ensure coordinates are within bounds
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(image.shape[1], x2), min(image.shape[0], y2)

        # Extract and process the person's ROI
        person_roi = image[y1:y2, x1:x2].copy()
        
        color_name, helmet_detect = draw_bounding_boxes_with_color_better(person_roi, model_path=helmet_model_path, dataset="None")

        helmet_detect_copy = helmet_detect.copy()

        if color_name=='yellow':
            count, mobile_detect = draw_bounding_boxes_mobile(reduce_glare(person_roi), helmet_detect_copy, model_path=mobile_model_path, dataset="None")

            if count!=0:
                #image[y1:y2, x1:x2] = mobile_detect
                hand_message, mobile_angle_detect = is_mobile_being_used(person_roi)
                
                full_image[y1:y2, x1:x2] = mobile_angle_detect

                message = hand_message
            else:
                full_image[y1:y2, x1:x2] = helmet_detect
                message = "Yellow Helmet Detected | Mobile Phone not Detected"
        else:
            full_image[y1:y2, x1:x2] = helmet_detect
            message = "Non-Yellow Helmet Detected"
        
        

    # Display the result
    cv2_imshow(full_image)
    print(message)
    return message, full_image, count

In [ ]:
person_model_path = "/kaggle/input/person-detec-roi/person_detect/best.pt"
helmet_model_path = "/kaggle/input/yolobestmodel/bestmodelyolo/best.pt"
mobile_model_path = "/kaggle/input/mobiledetect/mobile detect/best.pt"

# Telegram

In [ ]:
from datetime import datetime
import random

used_project_ids = set()

def generate_project_id():
    while True:
        number = random.randint(25000, 25999)  # Choose a range you prefer
        project_id = f"SC {number}"
        if project_id not in used_project_ids:
            used_project_ids.add(project_id)
            return project_id


used_ids = set()

def generate_camera_id():
    while True:
        cam_id = random.randint(1000, 9999)  # 4-digit ID
        if cam_id not in used_ids:
            used_ids.add(cam_id)
            return cam_id

from datetime import datetime

def workerdistract_info():
    camera_id = generate_camera_id()
    now_time = datetime.now()
    proj_id = generate_project_id()
    final_text = f"""Worker Distraction Alert
Mobile Phone in Use Exceeding 10 minutes 
Project ID: {proj_id}
Camera ID: {camera_id}
Date: {now_time}
    """
    return final_text

In [ ]:
import numpy as np
from PIL import Image
from io import BytesIO
from telegram import Bot
import asyncio

from PIL import Image
import cv2
from io import BytesIO

async def send_text_image_from_numpy_array(text=None, photo=None):
    bot = Bot(token='Put the telegram token')

    if text is not None:
        await bot.send_message(chat_id='Put the chat id', text=text)

    if photo is not None:
        try:
            # Resize (optional): downscale to reduce file size
            #photo = cv2.resize(photo, (512, 512), interpolation=cv2.INTER_AREA)

            # Convert to PIL
            img_rgb = cv2.cvtColor(photo, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(img_rgb)

            # Convert to RGB and compress to JPEG
            pil_image = pil_image.convert("RGB")
            with BytesIO() as image_io:
                pil_image.save(image_io, format='JPEG', quality=70, optimize=True)  # Lower quality = smaller size
                image_io.seek(0)

                # Check size before sending
                if image_io.getbuffer().nbytes > 10485760:
                    raise ValueError("Image still too large after compression.")

                await bot.send_photo(chat_id='Put the chat id', photo=image_io)

        except Exception as e:
            print(f"Error sending photo: {e}")

In [ ]:
import os
import cv2
import random

async def process_images(image_dir):

    for dirname, _, filenames in os.walk(image_dir):
        random.shuffle(filenames)  # Shuffle image order randomly

        for filename in filenames:
            try:
                image_path = os.path.join(dirname, filename)
                using_phone, imagee, count = hardhat_mobile_detect(
                    image_path, person_model_path, helmet_model_path, mobile_model_path
                )

                cv2.imwrite(filename, imagee)

                if using_phone=="Mobile Used":
                    await send_text_image_from_numpy_array(
                        text=workerdistract_info(), photo=imagee
                    )

            except Exception as e:
                print(f"Error processing {filename}: {e}")
                continue

# Inference

# SITE 1

In [ ]:
await process_images("/kaggle/input/datasets/milan400/site-sample-video-frames/sample_video_frames/site 1")

# SITE 2

In [ ]:
await process_images("/kaggle/input/datasets/milan400/site-sample-video-frames/sample_video_frames/site 2")

# SITE 3

In [ ]:
await process_images("/kaggle/input/datasets/milan400/site-sample-video-frames/sample_video_frames/site 3")